# Autoregressive context length — Tables 7, 8 and 9

4 examples x 7 context lengths x 5 seeds = **140 runs** on a Colab T4.

`L` is how many of the **200** context steps of the autoregressive channel the model may
see; the rest are zeroed. `L = 200` is the full context here, `L = 0` removes the
channel's content entirely.

**The harness was raised from `L_IN = 100` to `L_IN = 200` to make `L = 150` and `L = 200`
mean anything.** Upstream, `ar_len` is clamped to `L_IN`, so on the old harness those two
settings would have silently returned the `L = 100` run. Cell 4 asserts
`max(LENGTHS) == A.L_IN` so this cannot happen unnoticed. Because the window contents
change, **none of the earlier `results/` runs can be reused** — this sweep writes to
`results_lin200/`.

Three tables come from this single sweep:

| table | what it shows |
|---|---|
| **7** | absolute R², mean ± s.d. over the five seeds |
| **8** | paired change against `L = 100` **on the same seed** — split luck cancels |
| **9** | Table 7 calibrated onto Table 2 Arm 3, so that `L = 100` equals the benchmark |

**It need not finish in one session.** Every run writes its own JSON; re-running the sweep
cell skips whatever is already on disk. After a disconnect, re-run cells 1-4 then cell 5.


## 1 · GPU check

In [ ]:
import torch, platform
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.")
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| python", platform.python_version())

## 2 · Point at the data

Upload this folder to `MyDrive/colab_context` (keep `data/` intact), or edit `BASE`.

In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

# ── edit this one line if you put the folder somewhere else ──
BASE = "/content/drive/MyDrive/colab_context"

DATA = os.path.join(BASE, "data")
OUTD = os.path.join(BASE, "results_lin200")   # NOT results/: that sweep ran at L_IN=100
os.makedirs(OUTD, exist_ok=True)

DB_PATHS = {"sdof":    os.path.join(DATA, "sdof_eta60_n1500.npz"),
            "frame3":  os.path.join(DATA, "frame3_v3_arm_database.npz"),
            "tunnel":  os.path.join(DATA, "tunnel_moving_train_database.npz"),
            "turbine": os.path.join(DATA, "wt_v20_full.npz")}

missing = [k for k, p in DB_PATHS.items() if not os.path.exists(p)]
assert not missing, f"missing databases {missing} under {DATA}"
for k, p in DB_PATHS.items():
    print(f"  {k:8s} {os.path.getsize(p)/1e6:7.1f} MB")
print("\nresults ->", OUTD)

## 3 · The harness

In [ ]:
%%writefile context_gpu.py
"""Autoregressive-context harness — GPU.

Lifted verbatim from colab_arch/arch_backbone_colab.ipynb by make_colab_ablation.py,
with one change: SEED is a per-run argument threaded through prep() and run_one(),
so a seed selects both the 70/15/15 split and the init. Everything else — window
construction, normalisation, the free-running overlap-add rollout, the metrics — is
byte-identical to the backbone sweep, which is what makes the two tables comparable.

Context control (already present upstream):
    ar_len=L          keep only the last L of the L_IN autoregressive context steps,
                      zeroing the rest, in both the training windows and the rollout.
                      ar_len=L_IN (=200 here) is the full context; ar_len=0 removes it.
                      Values above L_IN are clamped, which is why L_IN was raised to 200.
"""
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
L_IN, L_OUT = 200, 100
L_ALL = L_IN + L_OUT
HID = 96
BATCH = 64
LR = 2e-3
WD = 4e-4

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DB: dict = {}          # example -> path to .npz, set by the notebook


# ─────────────────────────── data ───────────────────────────
def load_example(name):
    """-> u_nl, u_red, u_lin, q  each (N, T, S/Sq);  P (N, k);  fs"""
    if name == "sdof":
        d = np.load(DB[name])
        g = np.where(d["converged"])[0]
        u = d["u"][g][:, :, None]
        ul = d["u_lin"][g][:, :, None]
        q = d["F"][g][:, :, None]
        P = d["params"][g]
        fs = float(d["fs"])
        u_lin = ul                              # SDOF has no reduced model: u_red == u_lin
    else:
        d = np.load(DB[name], allow_pickle=True)
        tr = lambda a: np.transpose(a, (0, 2, 1)).astype(np.float32)
        u, ul, q = tr(d["u_nl"]), tr(d["u_red"]), tr(d["q_sensor"])
        u_lin = tr(d["u_lin"])
        P = d["params"]
        fs = 1.0 / float(d["dt"]) if "dt" in d.files else 1.0 / float(d["t"][1] - d["t"][0])
    return (u.astype(np.float32), ul.astype(np.float32), u_lin.astype(np.float32),
            q.astype(np.float32), P.astype(np.float64), fs)


# ─────────────────────── backbones ───────────────────────
class GRUNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.GRU(c, h, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.out_dim = h * 2
    def forward(self, x): return self.net(x)[0][:, -L_OUT:, :]


class LSTMNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.LSTM(c, h, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.out_dim = h * 2
    def forward(self, x): return self.net(x)[0][:, -L_OUT:, :]


class MLPNet(nn.Module):
    """Flatten the whole window -> MLP -> reshape. No temporal inductive bias."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.d = 64
        self.net = nn.Sequential(nn.Linear(L_ALL * c, 512), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(512, 512), nn.GELU(),
                                 nn.Linear(512, L_OUT * self.d))
        self.out_dim = self.d
    def forward(self, x):
        return self.net(x.flatten(1)).view(x.shape[0], L_OUT, self.d)


class TCNNet(nn.Module):
    """Dilated causal-style conv stack with residual connections."""
    def __init__(self, c, h=HID):
        super().__init__()
        layers, cin = [], c
        for dil in (1, 2, 4, 8, 16):
            layers.append(nn.Sequential(
                nn.Conv1d(cin, h, 3, padding=dil, dilation=dil), nn.GELU(),
                nn.Conv1d(h, h, 3, padding=dil, dilation=dil), nn.GELU()))
            cin = h
        self.blocks = nn.ModuleList(layers)
        self.out_dim = h
    def forward(self, x):
        z = x.transpose(1, 2)
        for i, b in enumerate(self.blocks):
            y = b(z)[:, :, :z.shape[2]]
            z = y if i == 0 else z + y
        return z.transpose(1, 2)[:, -L_OUT:, :]


class CNN1DNet(nn.Module):
    """Plain (non-dilated) 1-D conv stack."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(c, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU(),
            nn.Conv1d(h, h, 7, padding=3), nn.GELU())
        self.out_dim = h
    def forward(self, x): return self.net(x.transpose(1, 2)).transpose(1, 2)[:, -L_OUT:, :]


class ResBlock(nn.Module):
    def __init__(self, h, k=5):
        super().__init__()
        self.c1 = nn.Conv1d(h, h, k, padding=k // 2); self.b1 = nn.BatchNorm1d(h)
        self.c2 = nn.Conv1d(h, h, k, padding=k // 2); self.b2 = nn.BatchNorm1d(h)
    def forward(self, z):
        y = F.gelu(self.b1(self.c1(z)))
        y = self.b2(self.c2(y))
        return F.gelu(z + y)


class ResNetNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        self.stem = nn.Conv1d(c, h, 7, padding=3)
        self.blocks = nn.Sequential(*[ResBlock(h) for _ in range(4)])
        self.out_dim = h
    def forward(self, x):
        return self.blocks(F.gelu(self.stem(x.transpose(1, 2)))).transpose(1, 2)[:, -L_OUT:, :]


class TransformerNet(nn.Module):
    def __init__(self, c, h=HID):
        super().__init__()
        d = 128
        self.inp = nn.Linear(c, d)
        pe = torch.zeros(L_ALL, d)
        pos = torch.arange(L_ALL).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d, 4, dim_feedforward=256, dropout=0.1,
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=3)
        self.out_dim = d
    def forward(self, x):
        return self.enc(self.inp(x) + self.pe)[:, -L_OUT:, :]


class MambaLikeNet(nn.Module):
    """Mamba-style selective diagonal SSM, pure PyTorch.

    mamba_ssm's fused kernel is not used here, so this reimplements the core S6
    recurrence  h_t = exp(dt_t * A) h_{t-1} + dt_t * B_t * x_t ,  y_t = C_t * h_t
    with input-dependent dt, B, C and a diagonal A, evaluated by a log-space cumulative
    scan. Label results as 'Mamba-style', not the reference kernel.
    """
    def __init__(self, c, h=HID):
        super().__init__()
        self.h = h
        self.inp = nn.Linear(c, h)
        self.dt = nn.Linear(c, h)
        self.Bp = nn.Linear(c, h)
        self.Cp = nn.Linear(c, h)
        self.A_log = nn.Parameter(torch.log(torch.rand(h) * 0.5 + 0.5))
        self.conv = nn.Conv1d(h, h, 3, padding=1, groups=h)
        self.norm = nn.LayerNorm(h)
        self.out = nn.Linear(h, h)
        self.out_dim = h
    def forward(self, x):
        u = self.inp(x)
        u = self.conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)
        dt = F.softplus(self.dt(x))                        # (B,L,H) > 0
        A = -torch.exp(self.A_log)[None, None, :]          # (1,1,H) < 0
        logdec = dt * A                                    # log of decay, <= 0
        cum = torch.cumsum(logdec, dim=1)                  # (B,L,H)
        src = dt * self.Bp(x) * u
        # h_t = exp(cum_t) * sum_{s<=t} exp(-cum_s) * src_s   (clamped for stability)
        inner = torch.cumsum(src * torch.exp(-(cum - cum.max(dim=1, keepdim=True).values)
                                             .clamp(min=-30.0)), dim=1)
        h = torch.exp((cum - cum.max(dim=1, keepdim=True).values).clamp(min=-30.0)) * inner
        y = self.Cp(x) * h
        return self.out(self.norm(y))[:, -L_OUT:, :]


class EncDecNet(nn.Module):
    """GRU encoder -> repeated context -> GRU decoder over the output window."""
    def __init__(self, c, h=HID):
        super().__init__()
        self.enc = nn.GRU(c, h, num_layers=1, batch_first=True, bidirectional=True)
        self.dec = nn.GRU(h * 2 + c, h, num_layers=1, batch_first=True)
        self.out_dim = h
    def forward(self, x):
        _, hn = self.enc(x[:, :L_IN, :])
        ctx = torch.cat([hn[0], hn[1]], dim=-1).unsqueeze(1).expand(-1, L_OUT, -1)
        return self.dec(torch.cat([ctx, x[:, -L_OUT:, :]], dim=-1))[0]


ARCHS = {"gru": GRUNet, "lstm": LSTMNet, "mlp": MLPNet, "tcn": TCNNet,
         "cnn1d": CNN1DNet, "resnet": ResNetNet, "transformer": TransformerNet,
         "mamba": MambaLikeNet, "encdec": EncDecNet}

NICE = {"gru": "GRU", "transformer": "Transformer", "mamba": "Mamba",
        "encdec": "Encoder-decoder", "lstm": "LSTM", "mlp": "MLP", "tcn": "TCN",
        "resnet": "ResNet", "cnn1d": "1D CNN"}


# ─────────────────────── head + conditioning ───────────────────────
class FiLM(nn.Module):
    def __init__(self, n_static, n_time):
        super().__init__()
        h = max(n_static * 2, 64)
        self.f = nn.Sequential(nn.Linear(n_static, h), nn.ReLU(), nn.Linear(h, n_time * 2))
    def forward(self, x, s):
        g, b = self.f(s).unsqueeze(1).chunk(2, dim=-1)
        return (1.0 + g) * x + b


class NoFiLM(nn.Module):
    """Ablation: broadcast-concat the statics instead of modulating with them."""
    def __init__(self, n_static, n_time):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(n_static, 64), nn.ReLU(), nn.Linear(64, n_time))
    def forward(self, x, s): return x + self.f(s).unsqueeze(1)


class Head(nn.Module):
    def __init__(self, d, s, zero_init=True):
        super().__init__()
        self.a = nn.Sequential(nn.Linear(d, 64), nn.GELU())
        self.b = nn.Linear(64, s, bias=False)
        if zero_init:
            nn.init.zeros_(self.b.weight)      # start at Delta = 0  => Arm3 == Arm1
    def forward(self, z): return self.b(self.a(z))


class LoadSkip(nn.Module):
    """Per-sensor learned linear skip from the local load q to the local residual, with a
    gain that is a function of the static features. Zero-init so the recurrent path
    dominates early."""
    def __init__(self, n_static, sq, s):
        super().__init__()
        self.gain = nn.Sequential(nn.Linear(n_static, 64), nn.Tanh(), nn.Linear(64, s))
        nn.init.zeros_(self.gain[-1].weight); nn.init.zeros_(self.gain[-1].bias)
        self.proj = nn.Identity() if sq == s else nn.Linear(sq, s, bias=False)
    def forward(self, q_out, s_flat):
        return self.gain(s_flat).unsqueeze(1) * self.proj(q_out)


def tukey(n, a=0.3):
    w = np.ones(n, np.float32); k = int(a * n / 2)
    if k > 0:
        t = 0.5 * (1 - np.cos(np.pi * np.arange(k) / k)); w[:k] = t; w[-k:] = t[::-1]
    return w


# ─── window construction: identical for all 9 backbones, so build once per example ───
_PREP: dict = {}


def prep(example, train_frac=1.0, ar_len=None, seed=SEED):
    """Everything upstream of the model. Cached on (example, train_frac, ar_len, seed).
    The seed drives the 70/15/15 split, so each seed is a different split as well as a
    different init — matching how the release runs define a seed."""
    key = (example, train_frac, ar_len, seed)
    if key in _PREP:
        return _PREP[key]
    t0 = time.perf_counter()
    u_nl, u_red, u_lin, q, P, fs = load_example(example)
    N, T, S = u_nl.shape
    Sq = q.shape[2]
    step = max(L_OUT, T // 10)                       # ~10 windows per sample

    idx = np.random.default_rng(seed).permutation(N)
    n_tr, n_va = int(0.70 * N), int(0.15 * N)
    itr_full, iva, ite = idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]
    itr = itr_full[:max(1, int(round(len(itr_full) * train_frac)))]

    qs = q.std(1, keepdims=True) + 1e-12
    qn = (q / qs).astype(np.float32)
    rs = u_red.std(1, keepdims=True) + 1e-12
    rn = (u_red / rs).astype(np.float32)
    dl = ((u_lin - u_red) / rs).astype(np.float32)
    use_dlin = float(np.abs(dl).max()) > 1e-8
    resid = (u_nl - u_red).astype(np.float32)
    SIG = resid[itr].std(axis=(0, 1), keepdims=True) + 1e-12
    tgt = (resid / SIG).astype(np.float32)

    lp = np.log(np.clip(np.abs(P) + 1e-12, 1e-30, None))
    st = ((lp - lp[itr].mean(0)) / (lp[itr].std(0) + 1e-8)).astype(np.float32)
    NS = st.shape[1]

    tl = np.arange(L_ALL, dtype=np.float32) / fs
    ps = np.sin(2 * np.pi * 1.0 * tl)[None, :, None].astype(np.float32)
    pc = np.cos(2 * np.pi * 1.0 * tl)[None, :, None].astype(np.float32)

    AR_LEN = L_IN if ar_len is None else max(0, min(int(ar_len), L_IN))
    starts = [s_ for s_ in range(0, T, step) if s_ + L_ALL <= T + L_IN]
    z = np.zeros((L_IN, S), np.float32); zq = np.zeros((L_IN, Sq), np.float32)

    def build(ids):
        X, SS, Y = [], [], []
        for i in ids:
            qp = np.concatenate([zq, qn[i]]); rp = np.concatenate([z, rn[i]])
            dp = np.concatenate([z, dl[i]]);  tp = np.concatenate([z, tgt[i]])
            for s_ in starts:
                e = s_ + L_ALL
                up = np.concatenate([tp[s_:s_ + L_IN], np.zeros((L_OUT, S), np.float32)])
                if AR_LEN < L_IN:              # keep only the last AR_LEN steps of context
                    up = up.copy(); up[:L_IN - AR_LEN] = 0.0
                ch = [qp[s_:e], rp[s_:e]] + ([dp[s_:e]] if use_dlin else []) + \
                     [up, ps[0], pc[0]]
                X.append(np.concatenate(ch, 1)); SS.append(st[i]); Y.append(tp[s_ + L_IN:e])
        return (np.asarray(X, np.float32), np.asarray(SS, np.float32), np.asarray(Y, np.float32))

    Xtr, Str, Ytr = build(itr)
    Xva, Sva, Yva = build(iva)
    g = lambda a: torch.from_numpy(a).to(DEV)
    out = dict(Xtr=g(Xtr), Str=g(Str), Ytr=g(Ytr), Xva=g(Xva), Sva=g(Sva), Yva=g(Yva),
               C=Xtr.shape[2], NS=NS, S=S, Sq=Sq, T=T, N=N, fs=fs,
               itr=itr, iva=iva, ite=ite, use_dlin=use_dlin, AR_LEN=AR_LEN,
               qn=qn, rn=rn, dl=dl, st=st, SIG=SIG, ps=ps, pc=pc,
               u_nl=u_nl, u_red=u_red, n_win=len(starts),
               prep_sec=round(time.perf_counter() - t0, 1))
    _PREP.clear()                  # one example at a time: the GPU tensors are large
    _PREP[key] = out
    print(f"  prep {example}: N={N} T={T} S={S} channels={out['C']} statics={NS} "
          f"| train {len(itr)} val {len(iva)} test {len(ite)} "
          f"| {out['Xtr'].shape[0]} train windows ({out['n_win']}/sample) "
          f"| {out['prep_sec']}s")
    return out


def run_one(example, arch, epochs=10, train_frac=1.0, no_film=False, no_skip=False,
            no_zeroinit=False, ar_len=None, save_ckpt=None, seed=SEED):
    """Train one backbone on one example and score it on the TEST split. -> result dict"""
    torch.manual_seed(seed); np.random.seed(seed)
    if DEV.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    D = prep(example, train_frac, ar_len, seed)
    C, NS, S, Sq, T = D["C"], D["NS"], D["S"], D["Sq"], D["T"]

    film = (NoFiLM if no_film else FiLM)(NS, C).to(DEV)
    back = ARCHS[arch](C).to(DEV)
    head = Head(back.out_dim, S, zero_init=not no_zeroinit).to(DEV)
    mods = [film, back, head]
    skip = None
    if not no_skip:
        skip = LoadSkip(NS, Sq, S).to(DEV); mods.append(skip)
    mods = tuple(mods)

    def fwd(xb, sb):
        y = head(back(film(xb, sb)))
        return y if skip is None else y + skip(xb[:, -L_OUT:, :Sq], sb)
    par = [p for m in mods for p in m.parameters()]
    npar = sum(p.numel() for p in par)

    Xtr, Str, Ytr = D["Xtr"], D["Str"], D["Ytr"]
    Xva, Sva, Yva = D["Xva"], D["Sva"], D["Yva"]
    ntr = Xtr.shape[0]
    steps = max(1, ntr // BATCH)                        # == len(DataLoader(drop_last=True))
    opt = torch.optim.AdamW(par, lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs,
                                              steps_per_epoch=steps, pct_start=0.3)
    crit = nn.MSELoss()
    best, bstate, t0 = float("inf"), None, time.perf_counter()
    for ep in range(epochs):
        for m in mods: m.train()
        order = torch.randperm(ntr, device=DEV)
        for k in range(steps):
            j = order[k * BATCH:(k + 1) * BATCH]
            opt.zero_grad(set_to_none=True)
            loss = crit(fwd(Xtr[j], Str[j]), Ytr[j])
            loss.backward(); nn.utils.clip_grad_norm_(par, 1.0); opt.step(); sch.step()
        for m in mods: m.eval()
        tot, nva = 0.0, Xva.shape[0]
        with torch.no_grad():
            for k in range(0, nva, BATCH):
                sl = slice(k, min(k + BATCH, nva))
                tot += crit(fwd(Xva[sl], Sva[sl]), Yva[sl]).item() * (sl.stop - sl.start)
        v = tot / nva
        if v < best:
            best = v
            bstate = [{k: t.detach().clone() for k, t in m.state_dict().items()} for m in mods]
        print(f"    ep {ep+1:2d}/{epochs}  train {loss.item():.5f}  val {v:.5f}"
              f"{'  *' if v == best else ''}", flush=True)
    for m, s_ in zip(mods, bstate): m.load_state_dict(s_)
    wall = time.perf_counter() - t0

    # ── free-running overlap-add rollout on TEST ──
    for m in mods: m.eval()
    ite, hop, tap = D["ite"], max(1, L_OUT // 8), tukey(L_OUT)
    qn, rn, dl, st = D["qn"], D["rn"], D["dl"], D["st"]
    ps, pc, use_dlin, AR_LEN = D["ps"], D["pc"], D["use_dlin"], D["AR_LEN"]
    out = np.zeros((len(ite), T, S), np.float32)
    with torch.no_grad():
        for b0 in range(0, len(ite), 16):
            ci = ite[b0:b0 + 16]; nb = len(ci)
            qp = np.concatenate([np.zeros((nb, L_IN, Sq), np.float32), qn[ci]], 1)
            rp = np.concatenate([np.zeros((nb, L_IN, S), np.float32), rn[ci]], 1)
            dp = np.concatenate([np.zeros((nb, L_IN, S), np.float32), dl[ci]], 1)
            acc = np.zeros((nb, T + L_IN, S), np.float32)
            wg = np.zeros((nb, T + L_IN, 1), np.float32)
            est = np.zeros((nb, T + L_IN, S), np.float32)
            sb = torch.from_numpy(st[ci]).to(DEV)
            for s_ in range(0, T, hop):
                e = s_ + L_ALL
                if e > T + L_IN: break
                up = np.concatenate([est[:, s_:s_ + L_IN],
                                     np.zeros((nb, L_OUT, S), np.float32)], 1)
                if AR_LEN < L_IN: up[:, :L_IN - AR_LEN] = 0.0
                ch = [qp[:, s_:e], rp[:, s_:e]] + ([dp[:, s_:e]] if use_dlin else []) + \
                     [up, np.repeat(ps, nb, 0), np.repeat(pc, nb, 0)]
                xb = torch.from_numpy(np.concatenate(ch, 2)).to(DEV)
                pr = fwd(xb, sb).cpu().numpy()
                acc[:, s_ + L_IN:e] += pr * tap[None, :, None]
                wg[:, s_ + L_IN:e] += tap[None, :, None]
                sw = np.where(wg[:, s_ + L_IN:e] > 0, wg[:, s_ + L_IN:e], 1.0)
                est[:, s_ + L_IN:e] = acc[:, s_ + L_IN:e] / sw
            sw = np.where(wg > 0, wg, 1.0)
            out[b0:b0 + nb] = (acc / sw)[:, L_IN:L_IN + T]

    u_nl, u_red = D["u_nl"], D["u_red"]
    pred = u_red[ite] + out * D["SIG"]
    tr_ = u_nl[ite]
    r2 = float(1 - ((tr_ - pred) ** 2).sum() / (((tr_ - tr_.mean()) ** 2).sum() + 1e-12))
    r2b = float(1 - ((tr_ - u_red[ite]) ** 2).sum() / (((tr_ - tr_.mean()) ** 2).sum() + 1e-12))
    # unweighted mean over sensors, for the tunnel and the turbine where the sensors span
    # decades of energy and the pooled value hides the quiet ones
    per = [float(1 - ((tr_[:, :, s] - pred[:, :, s]) ** 2).sum()
                 / (((tr_[:, :, s] - tr_[:, :, s].mean()) ** 2).sum() + 1e-12))
           for s in range(S)]
    per1 = [float(1 - ((tr_[:, :, s] - u_red[ite][:, :, s]) ** 2).sum()
                  / (((tr_[:, :, s] - tr_[:, :, s].mean()) ** 2).sum() + 1e-12))
            for s in range(S)]
    pk_t, pk_p = np.max(np.abs(tr_), 1), np.max(np.abs(pred), 1)
    if save_ckpt:
        # atomic: a session killed mid-write must not leave a truncated .pt beside a
        # valid .json, which would read as a complete run
        torch.save({"film": film.state_dict(), "back": back.state_dict(),
                    "head": head.state_dict(),
                    "skip": (skip.state_dict() if skip is not None else None),
                    "arch": arch, "example": example, "SEED": seed, "C": C, "S": S,
                    "no_film": no_film, "no_skip": no_skip, "no_zeroinit": no_zeroinit,
                    "SIG": D["SIG"], "itr": D["itr"], "iva": D["iva"], "ite": D["ite"]},
                   save_ckpt + ".tmp")
        os.replace(save_ckpt + ".tmp", save_ckpt)
    return dict(example=example, arch=arch, R2=r2, R2_arm1=r2b, gain=r2 - r2b,
                R2_unweighted=float(np.mean(per)), R2_arm1_unweighted=float(np.mean(per1)),
                r2_per_sensor=per, val_mse=best, params=npar, sec=round(wall, 1),
                peak_err=float((np.abs(pk_t - pk_p) / (pk_t + 1e-30) * 100).mean()),
                n_train=len(D["itr"]), n_val=len(D["iva"]), n_test=len(ite),
                channels=C, epochs=epochs, train_frac=train_frac, seed=seed,
                device=DEV.type, gpu=(torch.cuda.get_device_name(0)
                                     if DEV.type == "cuda" else None))


In [ ]:
import importlib
import context_gpu as A
importlib.reload(A)

## 4 · Configuration

In [ ]:
import os, shutil, time

LOCAL = "/content/db"
os.makedirs(LOCAL, exist_ok=True)
t0 = time.perf_counter()
for k, p in DB_PATHS.items():
    dst = os.path.join(LOCAL, os.path.basename(p))
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(p):
        shutil.copy2(p, dst)
    A.DB[k] = dst
print(f"databases staged locally in {time.perf_counter()-t0:.0f}s")

EXAMPLES = ["sdof", "frame3", "tunnel", "turbine"]
LENGTHS  = [0, 12, 25, 50, 100, 150, 200]   # 200 == L_IN == the full context
SEEDS    = [42, 0, 1, 2, 3]
BACKBONE = "gru"
EPOCHS   = 10
SAVE_CKPT = False                    # 140 checkpoints is a lot of Drive; JSONs suffice

# ar_len is clamped to L_IN upstream, so any L above it would silently rerun the L_IN case
# and report a difference of exactly zero. Both halves of this assert matter.
assert max(LENGTHS) == A.L_IN, f"L_IN is {A.L_IN}; the full-context entry must equal it"
assert A.L_OUT == 100, f"L_OUT is {A.L_OUT}; the prediction horizon must not have moved"
print(f"harness L_IN={A.L_IN}  L_OUT={A.L_OUT}  L_ALL={A.L_ALL}")
N = len(EXAMPLES) * len(LENGTHS) * len(SEEDS)
print(f"\n{len(EXAMPLES)} examples x {len(LENGTHS)} lengths x {len(SEEDS)} seeds "
      f"= {N} runs, {EPOCHS} epochs, backbone {BACKBONE}")

## 5 · Run

Resumable at per-run granularity: a run counts as done only when its JSON parses **and**
carries the metric, writes are atomic, and re-running after completion is a no-op.


In [ ]:
import json, os, time, traceback

def key(ex, L, seed):
    return f"{ex}_L{L}_s{seed}"

def finished(ex, L, seed):
    jf = os.path.join(OUTD, key(ex, L, seed) + ".json")
    if not os.path.exists(jf):
        return False
    try:
        r = json.load(open(jf))
        return "R2" in r and "sec" in r
    except Exception:
        print(f"   corrupt {os.path.basename(jf)} -> will re-run")
        return False

def save_atomic(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=1); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)

t_start = time.perf_counter()
todo = [(e, L, s) for e in EXAMPLES for L in LENGTHS for s in SEEDS]
pre = [t for t in todo if finished(*t)]
print(f"{len(pre)}/{len(todo)} already done, {len(todo)-len(pre)} to go\n")
n_new = 0

for n, (ex, L, seed) in enumerate(todo, 1):
    tag = f"{ex}/L={L}/s{seed}"
    if finished(ex, L, seed):
        print(f"[{n:3d}/{len(todo)}] {tag:24s} SKIP (done)")
        continue
    print(f"[{n:3d}/{len(todo)}] {tag:24s} ...", flush=True)
    try:
        ck = os.path.join(OUTD, key(ex, L, seed) + ".pt") if SAVE_CKPT else None
        r = A.run_one(ex, BACKBONE, epochs=EPOCHS, seed=seed, ar_len=L, save_ckpt=ck)
    except KeyboardInterrupt:
        print("\ninterrupted -- re-run this cell to continue from here"); break
    except Exception:
        traceback.print_exc(); print(f"   !! {tag} FAILED -- continuing\n"); continue
    r["L"] = L
    save_atomic(r, os.path.join(OUTD, key(ex, L, seed) + ".json"))
    with open(os.path.join(OUTD, "progress.csv"), "a") as fh:
        fh.write(f"{ex},{L},{seed},{r['R2']:.6f},{r['R2_unweighted']:.6f},"
                 f"{r['R2_arm1']:.6f},{r['sec']:.1f},"
                 f"{time.strftime('%Y-%m-%d %H:%M:%S')}\n")
    n_new += 1
    el = time.perf_counter() - t_start
    rem = (len(todo) - len(pre) - n_new) * el / n_new
    print(f"   R2={r['R2']:+.4f}  unweighted={r['R2_unweighted']:+.4f}  {r['sec']:.0f}s"
          f"   | {n_new} this session in {el/60:.0f}m, ETA {rem/60:.0f}m\n", flush=True)

nd = sum(finished(*t) for t in todo)
print(f"{nd}/{len(todo)} complete. {(time.perf_counter()-t_start)/60:.0f} min this session.")
if nd < len(todo):
    print("Not finished -- re-run cells 1-4 then this cell to continue.")

## 6 · Tables 7 and 8

Table 7 is absolute R², mean ± s.d. over the seeds. Table 8 is the **paired** change against
`L = 100` computed seed by seed, so the split cancels and what is left is the effect of the
context length alone. A negative value means shortening the context degrades accuracy.

Metric per row follows the benchmark: pooled for SDOF and the frame, unweighted mean over
sensors for the tunnel and the turbine.


In [ ]:
import csv, glob, json, math, os
import statistics as st

rows = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(OUTD, "*_L*_s*.json")))]
D = {(r["example"], r["L"], r["seed"]): r for r in rows}
PICK = {"sdof": "R2", "frame3": "R2",
        "tunnel": "R2_unweighted", "turbine": "R2_unweighted"}
NICE = {"sdof": "Single-degree-of-freedom", "frame3": "Three-storey steel frame",
        "tunnel": "Buried tunnel", "turbine": "Wind turbine"}

def have(ex, L):
    return [D[(ex, L, s)] for s in SEEDS if (ex, L, s) in D]

# ── Table 7 ──
t7 = [["System"] + [f"L = {L}" for L in LENGTHS]]
print("=== Table 7: absolute R2, mean +/- s.d. over seeds ===")
print(f"{'System':26} " + "  ".join(f"{'L = '+str(L):>18}" for L in LENGTHS))
for ex in EXAMPLES:
    mk = PICK[ex]; line = [NICE[ex]]
    for L in LENGTHS:
        rs = have(ex, L)
        line.append(f"{st.mean([r[mk] for r in rs]):.4f} +/- "
                    f"{st.stdev([r[mk] for r in rs]):.4f}"
                    if len(rs) == len(SEEDS) else f"({len(rs)}/{len(SEEDS)} seeds)")
    t7.append(line)
    print(f"{line[0]:26} " + "  ".join(f"{c:>18}" for c in line[1:]))

# ── Table 8: paired against L = 100 ──
SHORT = [L for L in LENGTHS if L != max(LENGTHS)]
t8 = [["System"] + [f"L = {L}" for L in SHORT]]
res = []
for ex in EXAMPLES:
    mk = PICK[ex]; line = [NICE[ex]]; d0 = None
    for L in SHORT:
        d = [D[(ex, L, s)][mk] - D[(ex, max(LENGTHS), s)][mk]
             for s in SEEDS if (ex, L, s) in D and (ex, max(LENGTHS), s) in D]
        if len(d) < len(SEEDS):
            line.append(f"({len(d)}/{len(SEEDS)})"); continue
        m, sd = st.mean(d), st.stdev(d)
        t = m / (sd / math.sqrt(len(d))) if sd > 0 else 0.0
        star = "*" if abs(t) > 2.776 else ("+" if abs(t) > 2.132 else "")
        line.append(f"{m:+.4f}{star}")
        if L == 0: d0 = abs(m)
    res.append((d0 if d0 is not None else -1, line))
res.sort(key=lambda x: -x[0])                     # order by |effect| at L = 0
t8 += [line for _, line in res]
print("\n=== Table 8: paired change vs L = 100, same seed ===")
print(f"{'System':26} " + "  ".join(f"{'L = '+str(L):>12}" for L in SHORT))
for line in t8[1:]:
    print(f"{line[0]:26} " + "  ".join(f"{c:>12}" for c in line[1:]))
print("\n+ p<0.10, * p<0.05 (paired t, 4 dof). Negative = shortening the context hurts.")

for nm, tb in (("table7", t7), ("table8", t8)):
    p = os.path.join(OUTD, nm + ".csv")
    with open(p, "w", newline="") as fh:
        csv.writer(fh).writerows(tb)
    print("wrote", p)

## 7 · Table 9 — calibrated onto the benchmark

The sweep runs one common harness, so its absolute level differs from the tuned
per-example configurations of Table 2. Table 9 removes that offset by scaling the
**unexplained** variance so that `L = 100` reproduces Arm 3 of Table 2 exactly:

`rho_e = (1 - R2_bench,e) / (1 - mean R2_sweep,e,L=100)`, then
`R2_cal = 1 - rho_e * (1 - R2_sweep)`.

Two properties worth knowing before quoting it. The map is affine in the error energy, so
every paired difference is simply multiplied by `rho`; where `rho > 1` the calibration
**amplifies** the effect of context length and where `rho < 1` it **shrinks** it. And
because a t-statistic is scale-invariant, the significance markers are identical to
Table 8 — calibration changes the size of an effect, never whether it is detectable.

`L = 100` is the anchor because three of the four benchmarks (SDOF, tunnel, turbine) run
at `L_in = 100`. The steel frame benchmark runs at `L_IN = L_OUT = 150`, so its anchor is
a setting its own benchmark never used; set `ANCHOR = 150` below to change that.


In [ ]:
import csv, os
import statistics as st

# Arm 3 of Table 2 — the tuned per-example benchmark this sweep is calibrated onto.
BENCH  = {"sdof": 0.9123, "frame3": 0.8363, "turbine": 0.8631, "tunnel": 0.8704}
ANCHOR = 100          # the length forced to equal the benchmark
assert ANCHOR in LENGTHS

rho = {}
for ex in EXAMPLES:
    rs = have(ex, ANCHOR)
    assert len(rs) == len(SEEDS), f"{ex}: all seeds needed at L={ANCHOR} to calibrate"
    rho[ex] = (1 - BENCH[ex]) / (1 - st.mean([r[PICK[ex]] for r in rs]))

def cal(ex, x):
    return 1 - rho[ex] * (1 - x)

print("=== calibration factors, error-energy scaling onto Table 2 Arm 3 ===")
print(f"{'System':26}{'sweep':>10}{'benchmark':>11}{'rho':>8}   effect on differences")
for ex in EXAMPLES:
    m = st.mean([r[PICK[ex]] for r in have(ex, ANCHOR)])
    verb = "amplified" if rho[ex] > 1 else "shrunk"
    print(f"{NICE[ex]:26}{m:10.4f}{BENCH[ex]:11.4f}{rho[ex]:8.3f}   {verb} {rho[ex]:.2f}x")

t9 = [["System"] + [f"L = {L}" for L in LENGTHS]]
print("\n=== Table 9: calibrated absolute R2 (L = 100 reproduces the benchmark) ===")
print(f"{'System':26} " + "  ".join(f"{'L = '+str(L):>18}" for L in LENGTHS))
for ex in EXAMPLES:
    mk = PICK[ex]; line = [NICE[ex]]
    for L in LENGTHS:
        rs = have(ex, L)
        if len(rs) != len(SEEDS):
            line.append(f"({len(rs)}/{len(SEEDS)} seeds)")
            continue
        v = [cal(ex, r[mk]) for r in rs]
        line.append(f"{st.mean(v):.4f} +/- {st.stdev(v):.4f}")
    t9.append(line)
    print(f"{line[0]:26} " + "  ".join(f"{c:>18}" for c in line[1:]))
    got = st.mean([cal(ex, r[mk]) for r in have(ex, ANCHOR)])
    assert abs(got - BENCH[ex]) < 1e-9, f"{ex}: L={ANCHOR} must equal the benchmark"

p9 = os.path.join(OUTD, "table9_calibrated.csv")
with open(p9, "w", newline="") as fh:
    csv.writer(fh).writerows(t9)
print("\nwrote", p9)
print(f"L = {ANCHOR} reproduces Table 2 Arm 3 exactly, by construction.")

---

### Reproducibility

`progress.csv` is an append-only log of what completed when. `SAVE_CKPT` is off by default
here — 100 checkpoints is a lot of Drive space and the JSONs carry every metric — set it to
`True` in cell 4 if you want the weights.

### Provenance

The harness is lifted verbatim from `colab_arch/arch_backbone_colab.ipynb` and patched only
to thread a per-run seed through `prep()` and `run_one()`; the patch list is asserted in
`make_colab_ablation.py`, which this builder imports. `ar_len` is an upstream feature, not
something added here: it zeroes all but the last `AR_LEN` steps of the autoregressive
channel, in both the training windows and the free-running rollout.

Databases are the `release_webapp/` files copied unmodified.

Rebuild with `python3 make_colab_context.py` from the project root.

### Reading the two tables together

Table 7's ± is dominated by seed spread and will often exceed the differences between
columns; that is expected and is exactly why Table 8 exists. Quote Table 8 for any claim
about context length, and treat Table 7 as the level context.
